<a href="https://colab.research.google.com/github/MorreyB/PC-Free/blob/main/linux_server_3.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import os, subprocess, shutil, psutil, time, threading, torch
from google.colab import drive, output
from IPython.display import HTML, display

# 1. MOUNT GOOGLE DRIVE
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# 2. INSTALL DEPENDENCIES
print("Installing system dependencies and GPU acceleration tools...")
subprocess.run(["apt-get", "update"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "xfce4", "xfce4-goodies", "tightvncserver", "novnc", "python3-websockify", "sudo", "firefox", "libnvidia-gl-535", "libegl1", "libgl1-mesa-glx", "dbus-x11", "mesa-utils"], capture_output=True)

if not shutil.which("vglrun"):
    print("Installing VirtualGL...")
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run(["dpkg", "-i", "vgl.deb"], capture_output=True)
    subprocess.run(["apt-get", "install", "-f", "-y"], capture_output=True)

# 2.5 NVIDIA & EGL FIXES
print("Configuring NVIDIA GPU nodes and EGL backends...")
for i in range(10):
    if not os.path.exists(f'/dev/nvidia{i}'):
        subprocess.run(f'mknod -m 666 /dev/nvidia{i} c 195 {i}', shell=True)
if not os.path.exists('/dev/nvidiactl'):
    subprocess.run('mknod -m 666 /dev/nvidiactl c 195 255', shell=True)
if not os.path.exists('/dev/nvidia-uvm'):
    subprocess.run('mknod -m 666 /dev/nvidia-uvm c 243 0', shell=True)

# Create EGL vendor config so VirtualGL sees NVIDIA
os.makedirs('/usr/share/glvnd/egl_vendor.d/', exist_ok=True)
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    f.write('{\n    "file_format_version" : "1.0.0",\n    "ICD" : {\n        "library_path" : "libEGL_nvidia.so.0"\n    }\n}')

# Symlink libraries
subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libGL.so.1", shell=True)
subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("chmod -R 666 /dev/nvidia*", shell=True)

# 3. CLEANUP & VNC SETUP
print("Restarting desktop services...")
subprocess.run("pkill -9 -f Xtightvnc|websockify|cloudflared|dbus", shell=True, capture_output=True)
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
time.sleep(2)

os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nunset SESSION_MANAGER\nunset DBUS_SESSION_BUS_ADDRESS\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\nexport VGL_EGLDEVICE=0\nexport LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu/nvidia:/usr/lib64-nvidia:$LD_LIBRARY_PATH\nexport __GLX_VENDOR_LIBRARY_NAME=nvidia\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)

if not os.path.exists("/root/.vnc/passwd"):
    with open("vnc_pass", "w") as f: subprocess.run("echo password | vncpasswd -f", shell=True, stdout=f)
    subprocess.run("mv vnc_pass /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)

# 5. START
print("Starting Accelerated Desktop on :10...")
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

time.sleep(5)
url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not url.endswith('/'): url += '/'
final_link = f"{url}vnc.html?autoconnect=true&reconnect=true"
display(HTML(f'<div style="padding:10px; border:2px solid #4285f4; border-radius:8px;"><b>Desktop Ready:</b> <a href="{final_link}" target="_blank">{final_link}</a></div>'))
display(HTML(f'<iframe src="{final_link}" width="100%" height="800px" style="border:3px solid #4285f4; border-radius:8px;"></iframe>'))

Mounted at /content/drive
Installing system dependencies and GPU acceleration tools...
Configuring NVIDIA GPU nodes and EGL backends...
Restarting desktop services...
Starting Accelerated Desktop on :10...


/usr/local/lib/python3.12/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [ ]:
# @title VRAM Reservation (Adjusted for Display)
import torch, time, threading, gc

def hold_vram():
    global reserve_tensor
    try:
        # Force clear existing tensor if it exists
        if 'reserve_tensor' in globals():
            del reserve_tensor
        gc.collect()
        torch.cuda.empty_cache()
        time.sleep(5)

        # Reserve 9.0GB to be safe for EGL/VNC and Sober overhead
        elements = (9.0 * 1024**3) // 4
        print(f"--- VRAM Reserve Active ---")
        reserve_tensor = torch.zeros(int(elements), device='cuda')
        print(f"✅ 9.0GB reserved (High overhead left for GPU Desktop)")
        while True: time.sleep(60)
    except Exception as e:
        print(f"❌ Allocation failed: {e}")

vram_thread = threading.Thread(target=hold_vram, daemon=True)
vram_thread.start()

In [ ]:
import subprocess, time, os
from google.colab import output
from IPython.display import HTML, display

print("--- Accelerated Desktop Repair ---")
# 1. Kill everything
subprocess.run("pkill -9 Xtightvnc", shell=True)
subprocess.run("pkill -9 websockify", shell=True)
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10

# 2. Restart VNC
print("Restarting VNC Server...")
os.environ['USER'] = 'root'
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)

# 3. Restart Websockify
print("Restarting Proxy...")
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

time.sleep(5)
new_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not new_url.endswith('/'): new_url += '/'
link = f"{new_url}vnc.html?autoconnect=true&grab_keyboard=true&reconnect=true"

display(HTML(f'<div style="padding:10px; border:2px solid #4285f4; border-radius:8px;"><b>REPAIR COMPLETE</b><br>New Link: <a href="{link}" target="_blank">{link}</a></div>'))

--- Accelerated Desktop Repair ---
Restarting VNC Server...
Restarting Proxy...


In [ ]:
import os, subprocess, time
from IPython.display import HTML, display

print("--- High-Reliability Tunnel (Cloudflare) ---")
# 1. Install cloudflared if missing
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("Installing tunnel client...")
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared", shell=True)
    subprocess.run("chmod +x /usr/local/bin/cloudflared", shell=True)

# 2. Kill existing tunnels
subprocess.run("pkill -9 cloudflared", shell=True)

# 3. Start tunnel for noVNC port (6080)
print("Starting tunnel...")
with open("tunnel.log", "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:6080"], stdout=f, stderr=f)

# 4. Wait and extract URL
time.sleep(8)
cf_url = ""
if os.path.exists("tunnel.log"):
    with open("tunnel.log", "r") as f:
        import re
        log_content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if match:
            cf_url = match.group(0)

if cf_url:
    final_link = f"{cf_url}/vnc.html?autoconnect=true&reconnect=true"
    display(HTML(f'''<div style="padding:15px; border:3px solid #00ff00; border-radius:10px; background-color:#f0fff0;"><b>STABLE LINK READY:</b><br><a href="{final_link}" target="_blank" style="font-size:16px; color:#006400;">{final_link}</a></div>'''))
else:
    print("❌ Tunnel failed to start. Please check logs or re-run.")

--- High-Reliability Tunnel (Cloudflare) ---
Installing tunnel client...
Starting tunnel...


In [ ]:
import subprocess, os, time, re, sys
from google.colab import output
from IPython.display import HTML, display

print("--- Final UI & Tunnel Fix ---")

# 1. Total Cleanup
subprocess.run("pkill -9 -f xfce4|xfwm4|vnc|websockify|dbus|at-spi|cloudflared", shell=True)
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log /content/tunnel.log

# 2. Reset Xstartup
os.makedirs("/root/.vnc", exist_ok=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\nexport VGL_EGLDEVICE=0\nexport __GLX_VENDOR_LIBRARY_NAME=nvidia\nexport LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu/nvidia:$LD_LIBRARY_PATH\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)

# 3. Start VNC
print("Starting VNC Server...")
os.environ['USER'] = 'root'
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)

# 4. Start noVNC and Cloudflare Tunnel
print("Initializing Secure Tunnel...")
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
with open("tunnel.log", "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:6080"], stdout=f, stderr=f)

# 5. Wait for the Cloudflare URL
cf_url = None
print("Waiting for stable connection link...")
for i in range(30):
    time.sleep(2)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            log_data = f.read()
            # Use raw string r'' to ensure dots are treated as literals
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)
            if match:
                cf_url = match.group(0)
                break
    sys.stdout.write(f"\rPolling tunnel... ({i+1}/30)")
    sys.stdout.flush()

if cf_url:
    final_link = f"{cf_url}/vnc.html?autoconnect=true&reconnect=true"
    display(HTML(f'<div style="padding:15px; border:3px solid #00ff00; border-radius:10px; background-color:#f0fff0;"><b>FIX APPLIED:</b><br><a href="{final_link}" target="_blank" style="font-size:16px; font-weight:bold; color:#006400;">{final_link}</a></div>'))
else:
    print("\n❌ Tunnel failed. Check tunnel.log for details.")

--- Final UI & Tunnel Fix ---
Starting VNC Server...
Initializing Secure Tunnel...
Waiting for stable connection link...
Polling tunnel... (2/30)--- VRAM Reserve Active ---
✅ 9.0GB reserved (High overhead left for GPU Desktop)


In [ ]:
import subprocess
import os
import shutil
import socket

print("--- OpenGL Hardware Acceleration Check ---")

vgl_path = "/opt/VirtualGL/bin/vglrun"

if not os.path.exists(vgl_path):
    print(f"❌ VirtualGL was not found at {vgl_path}. Please re-run the Main Setup cell (1e439648).")
else:
    # Verify VNC is listening on Port 5910 (Display :10)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        vnc_alive = s.connect_ex(('127.0.0.1', 5910)) == 0

    if not vnc_alive:
        print("❌ VNC Server is not detected on :10. Please run/restart cell 1e439648 first.")
    else:
        # Environment for VirtualGL with EGL backend
        env = os.environ.copy()
        env.update({
            "DISPLAY": ":10",
            "VGL_DISPLAY": "egl",
            "LD_LIBRARY_PATH": "/usr/lib/x86_64-linux-gnu/nvidia:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu",
            "__NV_PRIME_RENDER_OFFLOAD": "1",
            "__GLX_VENDOR_LIBRARY_NAME": "nvidia"
        })

        try:
            print("Testing renderer via VirtualGL + EGL backend...")
            # Run glxinfo -B (Brief) to check the renderer
            result = subprocess.run([vgl_path, "glxinfo", "-B"], capture_output=True, text=True, env=env)
            output = result.stdout + result.stderr

            lines = output.splitlines()
            found_nvidia = False
            for line in lines:
                if any(x in line for x in ["OpenGL vendor string", "OpenGL renderer string", "Device", "direct rendering"]):
                    print(f"  > {line.strip()}")
                    if "NVIDIA" in line or "Tesla T4" in line:
                        found_nvidia = True

            if found_nvidia:
                print("\n✅ Hardware acceleration is ACTIVE (NVIDIA Tesla T4)")
                print("Apps launched via 'vglrun' on your desktop will now use the GPU.")
            else:
                print("\n❌ GPU not detected. VirtualGL may be falling back to software rendering.")
                print("Full Debug Output:\n", output)
        except Exception as e:
            print(f"\n❌ Diagnostic error: {e}")

--- OpenGL Hardware Acceleration Check ---
Testing renderer via VirtualGL + EGL backend...

❌ GPU not detected. VirtualGL may be falling back to software rendering.
Full Debug Output:
 name of display: :10
[VGL] ERROR: in getRBOContext--
[VGL]    133: Unexpected NULL condition



In [ ]:
!bash

bash: cannot set terminal process group (1784): Inappropriate ioctl for device
bash: no job control in this shell
/content# nvidia-smi
Thu Jun 18 11:05:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   10363MiB /  15360MiB |      0%      Default |
|    

In [2]:
# @title
import socket, subprocess

def run_diag():
    print("🔍 DIAGNOSTIC CHECK")
    # Check if port 5911 (VNC) is open
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        if s.connect_ex(('127.0.0.1', 5911)) == 0:
            print("✅ VNC Server (5911): LIVE")
        else:
            print("❌ VNC Server (5911): NOT RESPONDING")

    # Check if port 6083 (noVNC) is open
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        if s.connect_ex(('127.0.0.1', 6083)) == 0:
            print("✅ noVNC Proxy (6083): LIVE")
        else:
            print("❌ noVNC Proxy (6083): NOT RESPONDING")

    # Check for X11 lock files
    import os
    locks = [f for f in os.listdir('/tmp') if f.startswith('.X') and f.endswith('-lock')]
    if locks: print(f"⚠️ Found lock files: {locks}")

run_diag()

🔍 DIAGNOSTIC CHECK
❌ VNC Server (5911): NOT RESPONDING
❌ noVNC Proxy (6083): NOT RESPONDING
⚠️ Found lock files: ['.X10-lock']


### 🛠️ Alternative: Cloudflare Tunnel (Fixes Connection Failures)
Run this cell if the standard 'Desktop Ready' link fails to connect. This creates a secure tunnel directly from the VNC proxy to the web.

In [ ]:
# @title
import os, subprocess, time

# 1. Install cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("Installing cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 2. Launch Tunnel in background for the noVNC port (6083)
print("Starting Cloudflare Tunnel for port 6083...")
log_file = "cloudflared.log"
with open(log_file, "w") as f:
    process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:6083"],
                               stdout=f, stderr=f)

# 3. Extract the URL from logs
time.sleep(5)
cf_url = ""
for _ in range(10):
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            content = f.read()
            if "trycloudflare.com" in content:
                import re
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if match:
                    cf_url = match.group(0)
                    break
    time.sleep(2)

if cf_url:
    print(f"\n✨ CLOUDFLARE TUNNEL LIVE")
    print(f"Use this link: {cf_url}/vnc.html?autoconnect=true")
else:
    print("❌ Tunnel failed to start. Check cloudflared.log")

Starting Cloudflare Tunnel for port 6083...

✨ CLOUDFLARE TUNNEL LIVE
Use this link: https://owns-tampa-exposed-impact.trycloudflare.com/vnc.html?autoconnect=true


In [ ]:
# @title
import os, subprocess

print("🛑 Terminating all background desktop processes...")

# Kill VNC servers
subprocess.run(["pkill", "-9", "Xtightvnc"], capture_output=True)
# Kill noVNC proxies
subprocess.run(["pkill", "-9", "websockify"], capture_output=True)
# Kill Cloudflare tunnels
subprocess.run(["pkill", "-9", "cloudflared"], capture_output=True)

# Optional: Clear remaining lock files to ensure a clean restart later
!rm -rf /tmp/.X* /tmp/.X11-unix

print("✅ All active background processes have been stopped. System memory cleared.")

🛑 Terminating all background desktop processes...
✅ All active background processes have been stopped. System memory cleared.


In [ ]:
# @title
import torch
import time

# Quick check if CUDA is available
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print("Allocating 2GB of VRAM to show usage...")

    # Allocate roughly 2GB
    dummy_tensor = torch.empty((1024 * 512, 1024), device='cuda')

    print("VRAM Allocated. Check 'nvidia-smi' in your VNC terminal now.")
    print("This script will hold the memory for 30 seconds...")
    time.sleep(30)

    del dummy_tensor
    torch.cuda.empty_cache()
    print("Memory released.")
else:
    print("❌ GPU not detected by PyTorch. Please ensure you are using a GPU runtime (Runtime -> Change runtime type).")

✅ GPU Detected: Tesla T4
Allocating 2GB of VRAM to show usage...
VRAM Allocated. Check 'nvidia-smi' in your VNC terminal now.
This script will hold the memory for 30 seconds...
Memory released.


### Keyboard Passthrough Optimization
This cell disables XFCE system shortcuts that often interfere with web-based remote tools and virtual classrooms.

In [ ]:
# @title
import subprocess

def optimize_keyboard(username):
    # Commands to clear XFCE window manager shortcuts that interfere with browsers
    commands = [
        f"sudo -u {username} xfconf-query -c xfce4-keyboard-shortcuts -p /xfwm4/custom/<Alt>Tab -r",
        f"sudo -u {username} xfconf-query -c xfce4-keyboard-shortcuts -p /xfwm4/custom/<Super>Tab -r",
        f"sudo -u {username} xfconf-query -c xfce4-keyboard-shortcuts -p /commands/custom/override -n -t bool -s true"
    ]
    for cmd in commands:
        subprocess.run(cmd, shell=True)
    print(f"✅ Keyboard shortcuts optimized for {username}.")

# Apply to the persistent user
optimize_keyboard('drive_user')
# Apply to root
optimize_keyboard('root')

✅ Keyboard shortcuts optimized for drive_user.
✅ Keyboard shortcuts optimized for root.


In [ ]:
# @title
# CONNECTION REPAIR & STATUS
import os, subprocess, psutil
from google.colab import output

def check_and_repair():
    print("--- Connection Diagnostic ---")
    vnc_active = any("Xtightvnc" in p.name() for p in psutil.process_iter())
    web_active = any("websockify" in p.name() for p in psutil.process_iter())

    print(f"VNC Server Active: {'✅' if vnc_active else '❌'}")
    print(f"Proxy (Websockify) Active: {'✅' if web_active else '❌'}")

    if not vnc_active or not web_active:
        print("\nAttempting Repair...")
        # Restart root and drive_user specific proxies if missing
        subprocess.run(["pkill", "-f", "websockify"], capture_output=True)

        # Port 6080 for Root
        subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6080", "localhost:5901"])
        # Port 6083 for drive_user (default if only one user added)
        subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6083", "localhost:5903"])

        root_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
        user_url = output.eval_js("google.colab.kernel.proxyPort(6083)")
        print(f"\nNew Root Link: {root_url}vnc.html?autoconnect=true")
        print(f"New User Link: {user_url}vnc.html?autoconnect=true")
    else:
        print("\nProcesses look healthy. If you still can't connect, try refreshing the page or checking if your VPN is blocking Colab's proxy.")

check_and_repair()

--- Connection Diagnostic ---
VNC Server Active: ❌
Proxy (Websockify) Active: ✅

Attempting Repair...

New Root Link: https://6080-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.devvnc.html?autoconnect=true
New User Link: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.devvnc.html?autoconnect=true


In [ ]:
# @title
# DIAGNOSTIC: Check Firefox Installation
import subprocess
import os

print("--- Firefox Diagnostic ---")
path_check = subprocess.run(["which", "firefox"], capture_output=True, text=True)
if path_check.stdout:
    print(f"Location: {path_check.stdout.strip()}")
    version_check = subprocess.run(["firefox", "--version"], capture_output=True, text=True)
    print(f"Version: {version_check.stdout.strip() if version_check.stdout else 'Error running version check'}")
else:
    print("❌ Firefox binary NOT found in PATH.")

# Check if it's a snap shim (which fails in Colab)
if os.path.exists("/usr/bin/firefox"):
    with open("/usr/bin/firefox", "rb") as f:
        content = f.read(100)
        if b"snap" in content:
            print("⚠️ Warning: /usr/bin/firefox appears to be a snap shim. This will not work.")

--- Firefox Diagnostic ---
Location: /usr/bin/firefox
Version: Error running version check
⚠️ Warning: /usr/bin/firefox appears to be a snap shim. This will not work.


In [ ]:
# @title
# DIAGNOSTIC: Check Sober (Roblox) Installation
import subprocess
import os

print("--- Sober Diagnostic ---")
sober_bin = "/usr/local/bin/sober"

# 1. Check if file exists
if os.path.exists(sober_bin):
    print(f"✅ Binary Found: {sober_bin}")

    # 2. Check Permissions
    is_executable = os.access(sober_bin, os.X_OK)
    print(f"Executable Permission: {'✅ Yes' if is_executable else '❌ No'}")

    # 3. Check Version/Help (to see if it runs)
    try:
        # Some binaries use --version, others don't support it well
        res = subprocess.run([sober_bin, "--help"], capture_output=True, text=True, timeout=5)
        print("Binary Execution Test: ✅ Success (Help menu accessed)")
    except Exception as e:
        print(f"⚠️ Binary Execution Test: Failed to run with --help ({e})")

    # 4. Check for dependencies (ldd)
    print("\nChecking library dependencies (ldd):")
    ldd_res = subprocess.run(["ldd", sober_bin], capture_output=True, text=True)
    if "not found" in ldd_res.stdout:
        print("⚠️ Warning: Some missing libraries detected:")
        for line in ldd_res.stdout.splitlines():
            if "not found" in line:
                print(f"  - {line.strip()}")
    else:
        print("✅ All core shared libraries resolved.")
else:
    print(f"❌ Sober binary NOT found at {sober_bin}.")

--- Sober Diagnostic ---
✅ Binary Found: /usr/local/bin/sober
Executable Permission: ✅ Yes
⚠️ Binary Execution Test: Failed to run with --help ([Errno 8] Exec format error: '/usr/local/bin/sober')

Checking library dependencies (ldd):
✅ All core shared libraries resolved.


In [ ]:
import subprocess

print("--- Flatpak Sober Diagnostic ---")
# Check if Flatpak sees the installation
res = subprocess.run(["flatpak", "info", "org.vinegarhq.Sober"], capture_output=True, text=True)
if res.returncode == 0:
    print("✅ Sober is correctly installed in Flatpak.")
    print(res.stdout)
else:
    print("❌ Sober NOT found in Flatpak list. Attempting to verify flathub...")
    subprocess.run(["flatpak", "remotes"], capture_output=False)

print("\n--- Attempting Manual Execution Trace ---")
# This will show us exactly why it might be failing (missing drivers, dbus issues, etc.)
try:
    # We run with --help just to see if the container starts
    res_exec = subprocess.run(["flatpak", "run", "org.vinegarhq.Sober", "--help"], capture_output=True, text=True, timeout=10)
    print(f"Execution Output: {res_exec.stdout}")
    print(f"Execution Errors: {res_exec.stderr}")
except Exception as e:
    print(f"Execution failed: {e}")

--- Flatpak Sober Diagnostic ---
✅ Sober is correctly installed in Flatpak.

VinegarHQ & Sober contributors - Play, chat & explore on Roblox

          ID: org.vinegarhq.Sober
         Ref: app/org.vinegarhq.Sober/x86_64/stable
        Arch: x86_64
      Branch: stable
     Version: 1.7.0
     License: LicenseRef-proprietary=https://sober.vinegarhq.org/notice.txt
      Origin: flathub
  Collection: org.flathub.Stable
Installation: system
   Installed: 17.6 MB
     Runtime: org.gnome.Platform/x86_64/50
         Sdk: org.gnome.Sdk/x86_64/50

      Commit: 102964bde9e5c3f14288b75e0ff54777ae06fdea98be073d679b74fa3d9eccb8
      Parent: 141f90b706b55221a6b4706c87795b5a1c620f0b0e210748c2ef71e288f0f43c
     Subject: Update Sober to 2026-06-08_43d37ae (1.7.0 refresh) (90bad89c3dd2)
        Date: 2026-06-08 17:03:30 +0000


--- Attempting Manual Execution Trace ---
Execution Output: 
Execution Errors: error: Could not connect: No such file or directory



# Linux Remote Desktop with noVNC
This notebook sets up a lightweight XFCE desktop environment and accesses it via noVNC.

### Persistent Storage with Google Drive
By mounting Google Drive, we can store user home directories and configurations in a location that persists even if the Colab runtime is reset.

In [ ]:
# @title
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a base directory for our remote desktop data
PERSISTENT_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(PERSISTENT_BASE, exist_ok=True)
print(f"Persistent data will be stored in: {PERSISTENT_BASE}")

Mounted at /content/drive
Persistent data will be stored in: /content/drive/MyDrive/colab_desktop_data


In [ ]:
# @title
import subprocess
# Install XFCE desktop, VNC server, noVNC, and utilities to all users
print("Updating and installing system dependencies...")
subprocess.run(["apt-get", "update"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "xfce4", "xfce4-goodies", "tightvncserver", "novnc", "python3-websockify", "sudo", "curl", "wget"], capture_output=True)
print("Core desktop packages installed.")

Updating and installing system dependencies...
Core desktop packages installed.


In [ ]:
# @title
import subprocess
# Install Firefox
print("Installing Firefox...")
subprocess.run(["apt-get", "install", "-y", "firefox"], capture_output=True)
print("Firefox installation complete.")

Installing Firefox...
Firefox installation complete.


### Launch Firefox
You can launch Firefox from the terminal inside your VNC desktop by typing `firefox`, or run the cell below to trigger it from the notebook.

In [ ]:
# @title
# Launch Firefox on the root display (:1)
import os
subprocess.Popen(["sudo", "-u", "root", "DISPLAY=:1", "firefox"], env=os.environ)
print("Firefox launched on Display :1")

Firefox launched on Display :1


In [ ]:
# @title
import os
import subprocess

def create_firefox_shortcut(username):
    # Determine desktop path based on username
    if username == 'root':
        desktop_path = '/root/Desktop'
    else:
        desktop_path = f'/home/{username}/Desktop'

    # Ensure the Desktop directory exists
    subprocess.run(["sudo", "mkdir", "-p", desktop_path])

    shortcut_content = """[Desktop Entry]
Version=1.0
Type=Application
Name=Firefox Web Browser
Comment=Browse the World Wide Web
Exec=firefox %u
Terminal=false
Icon=firefox
Categories=Network;WebBrowser;
Path=
StartupNotify=true
"""

    file_path = os.path.join(desktop_path, 'firefox.desktop')

    # Write the shortcut using sudo to handle permissions correctly
    with open('temp_firefox.desktop', 'w') as f:
        f.write(shortcut_content)

    subprocess.run(["sudo", "mv", "temp_firefox.desktop", file_path])
    subprocess.run(["sudo", "chown", f"{username}:{username}", file_path])
    subprocess.run(["sudo", "chmod", "+x", file_path])
    print(f"✅ Firefox shortcut created for {username} at {file_path}")

# Create shortcuts for active users
create_firefox_shortcut('root')
if os.path.exists('/home/drive_user'):
    create_firefox_shortcut('drive_user')


✅ Firefox shortcut created for root at /root/Desktop/firefox.desktop
✅ Firefox shortcut created for drive_user at /home/drive_user/Desktop/firefox.desktop


In [ ]:
# @title
# Install NVIDIA drivers and utilities for the Linux environment
!apt-get install -y nvidia-utils-535

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following package was automatically installed and is no longer required:
  libfuse2
Use 'apt autoremove' to remove it.
The following additional packages will be installed:
  libnvidia-compute-535
Suggested packages:
  nvidia-driver-pinning-535 nvidia-driver-535
The following NEW packages will be installed:
  libnvidia-compute-535 nvidia-utils-535
0 upgraded, 2 newly installed, 0 to remove and 77 not upgraded.
Need to get 37.3 MB of archives.
After this operation, 177 MB of additional disk space will be used.
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  libnvidia-compute-535 535.309.01-1ubuntu1 [36.9 MB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  nvidia-utils-535 535.309.01-1ubuntu1 [382 kB]
Fetched 37.3 MB in 2s (21.8 MB/s)
Selecting previously unselected package libnvidia-compute-535:amd64.
(Reading database ..

### Verify GPU Connection
Once the desktop is loaded, open the **Terminal** inside the remote desktop and type:
`nvidia-smi`

This will show you the status of the Tesla T4 GPU assigned to this session.

In [ ]:
# @title
# Set up VNC server
import os
os.environ['USER'] = 'root'

# Initialize config
!mkdir -p ~/.vnc
!echo "password" | vncpasswd -f > ~/.vnc/passwd
!chmod 600 ~/.vnc/passwd

# Kill any existing sessions and start VNC server
!vncserver -kill :1 || true
!vncserver :1 -geometry 1280x720 -depth 24


Can't find file /root/.vnc/1720bdaf225b:1.pid
You'll have to kill the Xtightvnc process manually

xauth:  file /root/.Xauthority does not exist

New 'X' desktop is 1720bdaf225b:1

Creating default startup script /root/.vnc/xstartup
Starting applications specified in /root/.vnc/xstartup
Log file is /root/.vnc/1720bdaf225b:1.log



### Add Linux Login Screen (LightDM)
By default, VNC bypasses the login screen. The following code installs a display manager to require a Linux login within the session.

In [ ]:
# @title
# Install LightDM and the GTK Greeter
!apt-get install -y lightdm lightdm-gtk-greeter

# Configure Xvnc to launch the specific GTK greeter
with open('/root/.vnc/xstartup', 'w') as f:
    f.write('''#!/bin/sh
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
# Path to the installed GTK greeter
/usr/sbin/lightdm-gtk-greeter &
exec startxfce4
''')

!chmod +x /root/.vnc/xstartup
print("Login screen configuration updated with greeter. Please restart the session below.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
lightdm is already the newest version (1.30.0-0ubuntu5).
The following package was automatically installed and is no longer required:
  libfuse2
Use 'apt autoremove' to remove it.
The following additional packages will be installed:
  gnome-accessibility-themes gnome-themes-extra gnome-themes-extra-data
  gnome-themes-standard gtk2-engines-pixbuf
The following NEW packages will be installed:
  gnome-accessibility-themes gnome-themes-extra gnome-themes-extra-data
  gnome-themes-standard gtk2-engines-pixbuf lightdm-gtk-greeter
0 upgraded, 6 newly installed, 0 to remove and 76 not upgraded.
Need to get 2,490 kB of archives.
After this operation, 7,067 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 gnome-accessibility-themes all 3.28-1ubuntu3 [2,295 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 gnome-themes-extra-data all 3.28-1ub

### Create Linux User Account
Since the login screen requires a user to sign in, use the cell below to create a standard user account.

In [ ]:
# @title
# Create a new user named 'colab_user' with password 'password123'
!useradd -m -s /bin/bash colab_user
!echo "colab_user:password123" | chpasswd

# Add user to sudo group for administrative tasks
!usermod -aG sudo colab_user

print("User 'colab_user' created with password 'password123'.")
print("Now scroll down and run the 'Reconnect / Restart Session' cell.")

User 'colab_user' created with password 'password123'.
Now scroll down and run the 'Reconnect / Restart Session' cell.


### Set Up Multi-User VNC Instances
This cell initializes a private VNC configuration for `colab_user`. This allows this user to have an entirely separate desktop from the root user.

In [ ]:
# @title
import os

# Define the user and their VNC display number
USER_NAME = 'colab_user'
DISPLAY_NUM = ':2'
VNC_PORT = 5902
NOVNC_PORT = 6081

# Create .vnc directory for the user and set password
!sudo -u {USER_NAME} mkdir -p /home/{USER_NAME}/.vnc
!echo "password123" | sudo -u {USER_NAME} vncpasswd -f > /home/{USER_NAME}/.vnc/passwd

# FIX: Explicitly set ownership and strict permissions for the user's VNC directory
!sudo chown -R {USER_NAME}:{USER_NAME} /home/{USER_NAME}/.vnc
!sudo chmod 600 /home/{USER_NAME}/.vnc/passwd

# Create a private xstartup for this user
xstartup_path = f'/home/{USER_NAME}/.vnc/xstartup'
with open('temp_xstartup', 'w') as f:
    f.write('#!/bin/sh\nunset SESSION_MANAGER\nunset DBUS_SESSION_BUS_ADDRESS\n/usr/bin/startxfce4 &\n')

!sudo mv temp_xstartup {xstartup_path}
!sudo chown {USER_NAME}:{USER_NAME} {xstartup_path}
!sudo chmod +x {xstartup_path}

print(f"VNC environment fixed and initialized for {USER_NAME}. Now run the Launch cell below.")

VNC environment fixed and initialized for colab_user. Now run the Launch cell below.


### Launch Multi-User Sessions
Run this to start both the root desktop and the `colab_user` desktop simultaneously.

In [ ]:
# @title
import subprocess
from google.colab import output

# Kill existing to avoid conflicts
!vncserver -kill :1 || true
!vncserver -kill :2 || true
!pkill -f websockify

# Start Root Desktop (Display :1, Port 5901, noVNC 6080)
!USER=root vncserver :1 -geometry 1280x720 -depth 24
subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6080", "localhost:5901"])

# Start Colab_User Desktop (Display :2, Port 5902, noVNC 6081)
!sudo -u colab_user vncserver :2 -geometry 1280x720 -depth 24
subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6081", "localhost:5902"])

def get_url(port):
    url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
    if not url.endswith('/'): url += '/'
    return url + "vnc.html?autoconnect=true"

root_url = get_url(6080)
user_url = get_url(6081)

print(f"--- MULTI-USER ACCESS ---")
print(f"Root Desktop: {root_url}")
print(f"Colab_User Desktop: {user_url}")

Killing Xtightvnc process ID 21316

Can't find file /root/.vnc/1720bdaf225b:2.pid
You'll have to kill the Xtightvnc process manually


New 'X' desktop is 1720bdaf225b:1

Starting applications specified in /root/.vnc/xstartup
Log file is /root/.vnc/1720bdaf225b:1.log

A VNC server is already running as :2
--- MULTI-USER ACCESS ---
Root Desktop: https://6080-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true
Colab_User Desktop: https://6081-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true


### Dynamic User Creation
Run this cell to define the `add_vnc_user` function. You can call this multiple times to add different accounts with custom usernames and passwords.

In [ ]:
# @title
import subprocess
from google.colab import output

# Keep track of assigned ports to avoid overlaps
if 'user_registry' not in globals():
    user_registry = []

def add_vnc_user(username, password):
    display_num = len(user_registry) + 2
    vnc_port = 5900 + display_num
    novnc_port = 6080 + display_num - 1

    # 1. Create Linux User (if doesn't exist)
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", username], stderr=subprocess.DEVNULL)
    subprocess.run(["chpasswd"], input=f"{username}:{password}", encoding='ascii')
    subprocess.run(["usermod", "-aG", "sudo", username])

    # 2. Setup User VNC Directory
    user_home = f"/home/{username}"
    vnc_dir = f"{user_home}/.vnc"
    subprocess.run(["sudo", "-u", username, "mkdir", "-p", vnc_dir])

    # 3. Set VNC Password
    subprocess.run(["sudo", "-u", username, "vncpasswd", "-f"], input=password, encoding='ascii', stdout=open(f"{vnc_dir}/passwd", "w"))
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/passwd"])
    subprocess.run(["sudo", "chmod", "600", f"{vnc_dir}/passwd"])

    # 4. Create User Startup Script with GPU Support
    # We add NVIDIA paths to LD_LIBRARY_PATH for the user session
    xstartup_content = f"""#!/bin/sh
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
export PATH=/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH
/usr/bin/startxfce4 &
"""
    with open("temp_startup", "w") as f: f.write(xstartup_content)
    subprocess.run(["sudo", "mv", "temp_startup", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chmod", "+x", f"{vnc_dir}/xstartup"])

    # 5. Start/Restart VNC Server
    subprocess.run(["sudo", "-u", username, "vncserver", "-kill", f":{display_num}"], stderr=subprocess.DEVNULL)
    subprocess.run(["sudo", "-u", username, "vncserver", f":{display_num}", "-geometry", "1280x720", "-depth", "24"])

    # 6. Start noVNC Proxy
    subprocess.run(["pkill", "-f", f"websockify.*{novnc_port}"], stderr=subprocess.DEVNULL)
    subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", str(novnc_port), f"localhost:{vnc_port}"])

    # 7. Generate URL
    url = output.eval_js(f"google.colab.kernel.proxyPort({novnc_port})")
    if not url.endswith('/'): url += '/'
    full_user_url = url + "vnc.html?autoconnect=true"

    # Update registry if new
    if not any(u[0] == username for u in user_registry):
        user_registry.append((username, display_num, vnc_port, novnc_port))

    print(f"✅ User '{username}' updated with GPU support!")
    print(f"🔗 Access Desktop: {full_user_url}")

### Updated Dynamic User Creation (With Drive Persistence)
This version of the function symlinks the user's home directory to Google Drive to ensure files are saved permanently.

In [ ]:
# @title
def add_vnc_user_persistent(username, password, startup_script=None):
    display_num = len(user_registry) + 2
    vnc_port = 5900 + display_num
    novnc_port = 6080 + display_num - 1

    # 1. Create directory on Drive
    user_drive_path = os.path.join(PERSISTENT_BASE, username)
    os.makedirs(user_drive_path, exist_ok=True)

    # 2. Create User and Symlink
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", username], stderr=subprocess.DEVNULL)
    subprocess.run(["chpasswd"], input=f"{username}:{password}", encoding='ascii')

    user_local_home = f"/home/{username}"
    if not os.path.islink(user_local_home):
        if os.path.exists(user_local_home):
            subprocess.run(["cp", "-a", f"{user_local_home}/.", user_drive_path])
            subprocess.run(["rm", "-rf", user_local_home])
        subprocess.run(["ln", "-s", user_drive_path, user_local_home])

    # 3. Setup VNC
    vnc_dir = f"{user_local_home}/.vnc"
    subprocess.run(["sudo", "-u", username, "mkdir", "-p", vnc_dir])

    with open("vnc_pass_tmp", "w") as f:
        subprocess.run(["vncpasswd", "-f"], input=password, encoding='ascii', stdout=f)

    subprocess.run(["sudo", "mv", "vnc_pass_tmp", f"{vnc_dir}/passwd"])
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/passwd"])
    subprocess.run(["sudo", "chmod", "600", f"{vnc_dir}/passwd"])

    # 4. Custom Startup Logic
    base_script = f"""#!/bin/sh
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
export PATH=/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH
"""

    if startup_script:
        xstartup_content = base_script + startup_script + "\n/usr/bin/startxfce4 &\n"
    else:
        xstartup_content = base_script + "/usr/bin/startxfce4 &\n"

    with open("temp_startup", "w") as f: f.write(xstartup_content)
    subprocess.run(["sudo", "mv", "temp_startup", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chmod", "+x", f"{vnc_dir}/xstartup"])

    # 5. Launch
    subprocess.run(["sudo", "-u", username, "vncserver", "-kill", f":{display_num}"], stderr=subprocess.DEVNULL)
    subprocess.run(["sudo", "-u", username, "vncserver", f":{display_num}", "-geometry", "1280x720", "-depth", "24"])

    subprocess.run(["pkill", "-f", f"websockify.*{novnc_port}"], stderr=subprocess.DEVNULL)
    subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", str(novnc_port), f"localhost:{vnc_port}"])

    url = output.eval_js(f"google.colab.kernel.proxyPort({novnc_port})")
    if not url.endswith('/'): url += '/'

    if not any(u[0] == username for u in user_registry):
        user_registry.append((username, display_num, vnc_port, novnc_port))

    print(f"✅ User '{username}' initialized. Access: {url}vnc.html?autoconnect=true")

### Example: Create User with Custom Startup Script
This script will automatically launch Firefox and an XFCE Terminal upon login.

In [ ]:
# @title
my_startup_commands = """
# Launch Firefox in the background
firefox &
# Launch a terminal window
xfce4-terminal &
"""

# Create or update 'drive_user' with these startup commands
add_vnc_user_persistent("drive_user", "password123", startup_script=my_startup_commands)

✅ User 'drive_user' initialized. Access: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true


### Create a Persistent User
Use the function below to create a user whose data will stay on your Google Drive.

In [ ]:
# @title
# The previous error (FileNotFoundError) occurred because Python couldn't write directly to a sudo-owned directory.
# The function has been updated to use a 'temp-and-move' strategy to avoid this.
add_vnc_user_persistent("drive_user", "password123")

✅ User 'drive_user' created/updated with persistent Drive storage!
🔗 Access Desktop: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true


### Create a Persistent User
Use the function below to create a user whose data will stay on your Google Drive.

In [ ]:
# @title
# Re-run: Create a user named 'drive_user' with persistence
add_vnc_user_persistent("drive_user", "password123")

✅ User 'drive_user' created/updated with persistent Drive storage!
🔗 Access Desktop: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true


In [ ]:
# @title
import shutil

# Get disk usage for the persistent storage directory
total, used, free = shutil.disk_usage(PERSISTENT_BASE)

print(f"--- Google Drive Storage for {PERSISTENT_BASE} ---")
print(f"Total: {total // (2**30)} GB")
print(f"Used:  {used // (2**30)} GB")
print(f"Free:  {free // (2**30)} GB")

percent_used = (used / total) * 100
print(f"Usage: {percent_used:.2f}%")

--- Google Drive Storage for /content/drive/MyDrive/colab_desktop_data ---
Total: 15 GB
Used:  0 GB
Free:  14 GB
Usage: 0.31%


### Add Your Users Here
Call the function below to add as many accounts as you need.

In [ ]:
# @title
# Re-apply to existing users and add new ones if desired
add_vnc_user("bob", "password123")
add_vnc_user("alice", "securepass")

✅ User 'bob' updated with GPU support!
🔗 Access Desktop: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true
✅ User 'alice' updated with GPU support!
🔗 Access Desktop: https://6083-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev/vnc.html?autoconnect=true


### User Removal
Use this cell to define the function to remove a user and stop their active desktop processes.

In [ ]:
# @title
def remove_vnc_user(username):
    global user_registry
    # Find the user in the registry
    target_user = next((u for u in user_registry if u[0] == username), None)

    if target_user:
        username, display_num, vnc_port, novnc_port = target_user

        # 1. Kill the VNC server for that specific display
        subprocess.run(["sudo", "-u", username, "vncserver", "-kill", f":{display_num}"], stderr=subprocess.DEVNULL)

        # 2. Kill the websockify proxy for that specific port
        subprocess.run(["pkill", "-f", f"websockify.*{novnc_port}"], stderr=subprocess.DEVNULL)

        # 3. Remove from registry
        user_registry = [u for u in user_registry if u[0] != username]

        print(f"❌ User '{username}' removed and processes stopped.")
    else:
        print(f"☐ User '{username}' not found in registry.")

### Example: Remove a User
Uncomment the line below to remove a specific user.

In [ ]:
# @title
remove_vnc_user("bob")

❌ User 'bob' removed and processes stopped.


In [ ]:
# @title
# Start noVNC proxy
# We use port 6080 for noVNC which proxies to VNC port 5901
import subprocess

# Use Colab's output proxy to create a publicly accessible URL
from google.colab import output

# Start websockify in the background
subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6080", "localhost:5901"])

print("Click the link below to open your Remote Desktop:")
print(output.eval_js("google.colab.kernel.proxyPort(6080)"))

Click the link below to open your Remote Desktop:
https://6080-gpu-t4-s-kkb-usw4b1-zevlmuxgxw1p-b.us-west4-1.prod.colab.dev


### Integrated Remote Desktop
Run the cell below to display the desktop interface directly in the notebook.

**Note:** For better performance and to avoid browser permission issues, it is highly recommended to open the **Direct Link** generated below in a new browser tab.

In [ ]:
# @title
from IPython.display import HTML, display
from google.colab import output

# Get the proxied URL
base_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not base_url.endswith('/'): base_url += '/'
full_url = f"{base_url}vnc.html?autoconnect=true"

# Use a raw HTML string and display it specifically as 'text/html'
html_code = f'''
<iframe
    src="{full_url}"
    width="100%"
    height="720px"
    style="border:none;"
    allow="fullscreen; clipboard-read; clipboard-write;"
></iframe>
'''

display(HTML(html_code))
print(f"Direct Link: {full_url}")

Direct Link: https://6080-gpu-t4-s-kkb-use1c1-2q27v7z1llldq-c.us-east1-1.prod.colab.dev/vnc.html?autoconnect=true


### Interactive Linux Terminal
You can use the cell below to run any Linux command directly. For a full interactive shell, use the Terminal icon in the Colab left-hand sidebar.

In [ ]:
# @title
import os
import sys

def run_terminal():
    print("Interactive Terminal (type 'exit' to stop)")
    while True:
        cwd = os.getcwd()
        cmd = input(f"root@colab:{cwd}$ ")
        if cmd.lower() in ['exit', 'quit']:
            break
        if cmd.startswith('cd '):
            try:
                os.chdir(cmd[3:].strip())
            except Exception as e:
                print(e)
        else:
            os.system(cmd)

run_terminal()

Interactive Terminal (type 'exit' to stop)
root@colab:/content$ sudo install firefox
root@colab:/content$ sudo install firefox
root@colab:/content$ exit


In [ ]:
# @title
import os, subprocess, shutil, psutil, time, threading, torch, socket
from google.colab import drive, output
from IPython.display import HTML, display

# 1. MOUNT DRIVE
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# 2. INSTALLATION CHECK
if not shutil.which('tightvncserver'):
    print("Installing dependencies...")
    subprocess.run(["apt-get", "update"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "xfce4", "xfce4-goodies", "tightvncserver", "novnc", "python3-websockify", "sudo", "firefox"], capture_output=True)

# 3. NUCLEAR CLEANUP (Force-clearing stale locks)
print("Nuclear cleanup of X11 and VNC...")
subprocess.run(["pkill", "-9", "Xtightvnc"], capture_output=True)
subprocess.run(["pkill", "-9", "websockify"], capture_output=True)
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log /home/drive_user/.vnc/*.pid /home/drive_user/.vnc/*.log
os.makedirs('/tmp/.X11-unix', exist_ok=True)
subprocess.run(["chmod", "1777", "/tmp/.X11-unix"])

# 4. SETUP & LAUNCH
def launch_desktop(username, password, display_num, vnc_port, novnc_port):
    u_home = f"/home/{username}"
    u_drive = os.path.join(P_BASE, username)
    os.makedirs(u_drive, exist_ok=True)
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", username], stderr=subprocess.DEVNULL)
    subprocess.run(["chpasswd"], input=f"{username}:{password}", encoding='ascii')

    if not os.path.islink(u_home):
        if os.path.exists(u_home): shutil.rmtree(u_home)
        subprocess.run(["ln", "-s", u_drive, u_home])

    vnc_dir = f"{u_home}/.vnc"
    os.makedirs(vnc_dir, exist_ok=True)
    with open("vnc_pass", "w") as f: subprocess.run(["vncpasswd", "-f"], input=password, encoding='ascii', stdout=f)
    subprocess.run(["sudo", "mv", "vnc_pass", f"{vnc_dir}/passwd"])
    subprocess.run(["sudo", "chmod", "600", f"{vnc_dir}/passwd"])
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/passwd"])

    with open("xstart", "w") as f:
        f.write(f"#!/bin/sh\nunset SESSION_MANAGER\nunset DBUS_SESSION_BUS_ADDRESS\n/usr/bin/startxfce4 &\n")
    subprocess.run(["sudo", "mv", "xstart", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chmod", "+x", f"{vnc_dir}/xstartup"])
    subprocess.run(["sudo", "chown", f"{username}:{username}", f"{vnc_dir}/xstartup"])

    print(f"Starting 1080p VNC on Display :{display_num}...")
    # UPDATED: Changed geometry to 1920x1080
    subprocess.run(["sudo", "-u", username, "vncserver", f":{display_num}", "-geometry", "1920x1080", "-depth", "24", "-localhost", "no"])

    # Verification Loop
    for i in range(15):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', vnc_port)) == 0:
                print(f"✅ VNC Server is UP on port {vnc_port}")
                break
        time.sleep(1)

    subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", str(novnc_port), f"127.0.0.1:{vnc_port}"])

launch_desktop("drive_user", "password123", 11, 5911, 6083)

time.sleep(5)
url = output.eval_js("google.colab.kernel.proxyPort(6083)")
if not url.endswith('/'): url += '/'
final_link = f"{url}vnc.html?autoconnect=true"
print(f"\n🚀 1080p DESKTOP READY\nURL: {final_link}")
display(HTML(f'<iframe src="{final_link}" width="100%" height="900px" style="border:none;"></iframe>'))

Mounted at /content/drive
Nuclear cleanup of X11 and VNC...
Starting 1080p VNC on Display :11...

🚀 1080p DESKTOP READY
URL: https://6083-gpu-t4-s-kkb-use1c1-2q27v7z1llldq-c.us-east1-1.prod.colab.dev/vnc.html?autoconnect=true


/usr/local/lib/python3.12/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Reconnect / Restart Session
If the desktop screen is blank or you are signed out, run the cell below to restart the VNC display and the proxy tunnel.

In [ ]:
# @title
# Set user environment
import os
import subprocess
from google.colab import output
os.environ['USER'] = 'root'

# Restart VNC Server on Display :1
subprocess.run(["vncserver", "-kill", ":1"], stderr=subprocess.DEVNULL)
subprocess.run(["vncserver", ":1", "-geometry", "1280x720", "-depth", "24"])

# Restart Websockify Proxy
subprocess.run(["pkill", "-f", "websockify"])
subprocess.Popen(["websockify", "--web", "/usr/share/novnc/", "6080", "localhost:5901"])

# Provide fresh link
base_url = output.eval_js("google.colab.kernel.proxyPort(6080)")
if not base_url.endswith('/'): base_url += '/'
full_url = f"{base_url}vnc.html?autoconnect=true"
print(f"Desktop Restarted with GPU Support. Direct Link: {full_url}")

Desktop Restarted with GPU Support. Direct Link: https://6080-gpu-t4-s-kkb-use1c1-2q27v7z1llldq-c.us-east1-1.prod.colab.dev/vnc.html?autoconnect=true


### Troubleshooting Logs
Run this cell to view the VNC server logs. This will help identify if `lightdm-greeter` or `xfce4` failed to start.

In [ ]:
# @title
import os
# Find the latest log file
log_dir = os.path.expanduser('~/.vnc')
logs = [f for f in os.listdir(log_dir) if f.endswith('.log')]
if logs:
    latest_log = os.path.join(log_dir, sorted(logs)[-1])
    print(f"--- Showing contents of {latest_log} ---")
    with open(latest_log, 'r') as f:
        print(f.read())
else:
    print("No log files found.")

--- Showing contents of /root/.vnc/1720bdaf225b:1.log ---
17/06/26 11:13:24 Xvnc version TightVNC-1.3.10
17/06/26 11:13:24 Copyright (C) 2000-2009 TightVNC Group
17/06/26 11:13:24 Copyright (C) 1999 AT&T Laboratories Cambridge
17/06/26 11:13:24 All Rights Reserved.
17/06/26 11:13:24 See http://www.tightvnc.com/ for information on TightVNC
17/06/26 11:13:24 Desktop name 'X' (1720bdaf225b:1)
17/06/26 11:13:24 Protocol versions supported: 3.3, 3.7, 3.8, 3.7t, 3.8t
17/06/26 11:13:24 Listening for VNC connections on TCP port 5901
Font directory '/usr/share/fonts/X11/75dpi/' not found - ignoring
Font directory '/usr/share/fonts/X11/100dpi/' not found - ignoring
/root/.vnc/xstartup: 4: /usr/sbin/lightdm-greeter: not found
/usr/bin/startxfce4: X server already running on display :1

(xfwm4:17032): xfwm4-WARNING **: 11:13:25.488: XSync extension too old (3.0).

(xfwm4:17032): xfwm4-WARNING **: 11:13:25.488: The display does not support the XRender extension.

(xfwm4:17032): xfwm4-WARNING **: 11

### Alternative: Direct XFCE Start
If LightDM continues to be problematic in this environment, we can revert the `xstartup` to launch XFCE directly. You can still switch users inside the desktop by using the log-out menu.

In [ ]:
# @title
# Update root xstartup to include GPU acceleration paths
with open('/root/.vnc/xstartup', 'w') as f:
    f.write('''#!/bin/sh
unset SESSION_MANAGER
unset DBUS_SESSION_BUS_ADDRESS
# GPU Support for Root
export PATH=/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH
/usr/bin/startxfce4 &
''')

!chmod +x /root/.vnc/xstartup
print("Root configuration updated with GPU support. Restart the root session to apply.")

Root configuration updated with GPU support. Restart the root session to apply.
